# WPO 2: MDP & Q-Learning

## Exercise 1

Interact with the Frozen Lake environment from gymnasium. More information can be found on the Gymnasium website: https://gymnasium.farama.org/environments/toy_text/frozen_lake/.

*Description*:
Frozen lake involves crossing a frozen lake from start to goal without falling into any holes by walking over the frozen lake. The player may not always move in the intended direction due to the slippery nature of the frozen lake.


Tasks:

1) Define a policy that is a list of actions (LEFT, DOWN, RIGHT, UP) you believe will achieve highest return.
2) Observe and plot the gathered return per episode for your policy. Is there another sequence that would perform better given the stochastic nature of certain states?

In [1]:
import gymnasium as gym
import numpy as np
import random
from enum import Enum

In [2]:
class Actions(Enum):
    LEFT = 0
    DOWN = 1
    RIGHT = 2
    UP = 3

def print_action(action: Actions) -> str:
    if action == Actions.LEFT:
        return '<'
    elif action == Actions.DOWN:
        return 'v'
    elif action == Actions.RIGHT:
        return '>'
    elif action == Actions.UP:
        return '^'

In [3]:
# Define your policy as a list or a function, or whatever you want
from itertools import cycle 
policy_iter = cycle([Actions.RIGHT, Actions.DOWN, Actions.LEFT, Actions.UP])
policy = lambda state: next(policy_iter)


In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
env = gym.make(
    'FrozenLake-v1',
    is_slippery=True,
    map_name="4x4",  # change the map_name to "8x8" or None. Does your policy still work?
    #render_mode="rgb_array" ## Uncomment to see the environment visually
)
# Test out random agent on the evironment
terminated, truncated = False, False
state, _ = env.reset()

while not (terminated or truncated):
    action = policy(state)
    state, reward, terminated, truncated, _ = env.step(action.value)
    img = env.render()
    if img is not None:
        plt.imshow(img)
        plt.axis('off')
        plt.show()
    print(state, action, reward)
    

4 Actions.RIGHT 0
4 Actions.DOWN 0
4 Actions.LEFT 0
0 Actions.UP 0
4 Actions.RIGHT 0
5 Actions.DOWN 0


/home/andrea/Documents/PhD/RL_course/WP0s/venv/lib/python3.10/site-packages/gymnasium/envs/toy_text/frozen_lake.py:353: UserWarning: WARN: You are calling render method without specifying any render mode. You can specify the render_mode at initialization, e.g. gym.make("FrozenLake-v1", render_mode="rgb_array")
  gym.logger.warn(


In [5]:
def validate(policy, env, n_episodes: int):
    total_reward = 0
    for _ in range(n_episodes):
        terminated, truncated = False, False
        state, _ = env.reset()
        while not (terminated or truncated):
            action = policy(state)
            state, reward, terminated, truncated, _ = env.step(action.value)
            total_reward += reward
    print(f"Your policy has an average reward of {total_reward/n_episodes} over {n_episodes} episodes")

validate(policy, env, 1000)

Your policy has an average reward of 0.01 over 1000 episodes


## Exercise 2

Now implement the Q-learning algorithm and use it to train a policy with. Vary the epsilon valiue and observe the difference in learning by plotting the episodic return. After training, implement a validation method that exploits the learned policy and plot report on its performance.

In [ ]:
from typing import Dict


class QLearning:

    def __init__(self, env: gym.Env, epsilon: float):
        self.env = gym.make('FrozenLake-v1',is_slippery=True)
        self.epsilon = epsilon
        self.learning_rate = 0.1
        self.discount_factor = 0.9
        self.qtable = np.zeros((self.env.observation_space.n, self.env.action_space.n))

    def train(self, n_episodes: int):
        
        for _ in range(n_episodes):
            terminated, truncated = False, False
            state, _ = self.env.reset()
            while not (terminated or truncated):
                action = self.policy(state, greedy=False)
                next_state, reward, terminated, truncated, _ = self.env.step(action.value)
                # Q-learning update
                best_next_action = np.argmax(self.qtable[next_state])
                td_target = reward + self.discount_factor * self.qtable[next_state][best_next_action]
                self.qtable[state][action.value] += self.learning_rate * (td_target - self.qtable[state][action.value]) # TD error update
                state = next_state

    def policy(self,state: int, greedy: bool = True) -> Actions:
        if greedy or random.uniform(0, 1) > self.epsilon:
            action_idx = np.argmax(self.qtable[state])
            return Actions(action_idx)
        else:
            return Actions(random.randint(0, self.env.action_space.n - 1))
    
    def optimal_policy(self) -> Dict[int, Actions]:
        best_actions = self.qtable.argmax(axis=1)
        return {state: Actions(action) for state, action in enumerate(best_actions)}

    def print_optimal_action_map(self):
        optimal_policy = self.optimal_policy()
        action_map = [print_action(optimal_policy[state]) if self.qtable[state].any() else "." for state in range(self.env.observation_space.n)]
        row_dim = int(np.sqrt(self.env.observation_space.n))
        print("Optimal Action Map:")
        print('-'*(row_dim*4+2))
        for i in range(row_dim):
            print('| '+' | '.join(action_map[i*row_dim:(i+1)*row_dim])+' |')
            print('-'*(row_dim*4+2))
        print()


In [7]:
agent = QLearning(env, epsilon=0.4)
agent.train(n_episodes=10000)

validate(agent.policy, agent.env, n_episodes=1000)

Your policy has an average reward of 0.392 over 1000 episodes


In [8]:
optimal_policy = agent.optimal_policy()
print("Optimal Policy: \n", optimal_policy)

agent.print_optimal_action_map()

print(agent.qtable)

Optimal Policy: 
 {0: <Actions.RIGHT: 2>, 1: <Actions.UP: 3>, 2: <Actions.RIGHT: 2>, 3: <Actions.UP: 3>, 4: <Actions.LEFT: 0>, 5: <Actions.LEFT: 0>, 6: <Actions.LEFT: 0>, 7: <Actions.LEFT: 0>, 8: <Actions.UP: 3>, 9: <Actions.DOWN: 1>, 10: <Actions.LEFT: 0>, 11: <Actions.LEFT: 0>, 12: <Actions.LEFT: 0>, 13: <Actions.RIGHT: 2>, 14: <Actions.RIGHT: 2>, 15: <Actions.LEFT: 0>}
Optimal Action Map:
------------------
| > | ^ | > | ^ |
------------------
| < | . | < | . |
------------------
| ^ | v | < | . |
------------------
| . | > | > | . |
------------------

[[0.06141027 0.06390332 0.0668499  0.05822968]
 [0.03980038 0.04368559 0.03780198 0.06077805]
 [0.07551231 0.06466447 0.07623888 0.04900047]
 [0.03399759 0.02661706 0.03076463 0.05320134]
 [0.08276119 0.07328755 0.05207273 0.03785791]
 [0.         0.         0.         0.        ]
 [0.07322744 0.07128658 0.07106971 0.01855071]
 [0.         0.         0.         0.        ]
 [0.08701518 0.12329815 0.08595181 0.15436193]
 [0.20419481 0